In [ ]:
# 1 Crear 2000 pedidos usando customer_id
# 2 Crear unas 4500 líneas usando product_id
# 3 Calcular pagos según el total de cada pedido
# 4 Crear reseñas solo para productos entregados
# 5 Validar fechas, importes y claves foráneas (foreign keys)

import pandas as pd            
import numpy as np              
from faker import Faker          
from datetime import timedelta   

fake = Faker() # Inicializamos Faker para generar datos fake

np.random.seed(42) # resultados reproducibles

# CARGA DE DATOS DE ENG A

customers_df = pd.read_csv("customers.csv")

products_df = pd.read_csv("products.csv")

# VALIDACIONES DE DATOS MAESTROS 

assert customers_df["customer_id"].is_unique # Verificamos que no haya IDs duplicados en clientes

assert products_df["product_id"].is_unique # Verificamos que no haya IDs duplicados en productos

assert len(customers_df) >= 500 # al menos 500 clientes

assert len(products_df) >= 70 # al menos 70 productos

assert (products_df["stock"] >= 0).all() # no stock negativo

# FILTRAR PRODUCTOS VÁLIDOS

# Solo usamos productos:
# - disponibles (is_available = True)
# - con stock positivo
valid_products = products_df[
    (products_df["is_available"] == True) &
    (products_df["stock"] > 0)
]

# GENERACIÓN DE DATOS TRANSACCIONALES

# 1 Generamos la tabla de pedidos (orders)
# - Se crean 2000 pedidos
# - Cada pedido tendrá un customer_id existente
orders_df = generate_orders(customers_df, n_orders=2000)


# 2 Generamos líneas de pedido (order_items)
# - Usa pedidos ya creados
# - Usa productos válidos
# - Genera aprox 4500 líneas
# - Cada línea conecta order_id con product_id
order_items_df = generate_order_items(
    orders_df,
    valid_products,
    target_lines=4500
)


# 3 Generamos pagos (payments)
# - Cada pedido tiene uno o más pagos
# - El importe debe coincidir con la suma de sus líneas
payments_df = generate_payments(
    orders_df,
    order_items_df
)


# 4 Generamos reseñas (reviews)
# - Solo para pedidos entregados
# - Aproximadamente el 35% de las líneas
# - Cada review se asocia a order_item_id (no a order completo)
reviews_df = generate_reviews(
    orders_df,
    order_items_df,
    review_rate=0.35
)

# ---------------------------------------
# 5 VALIDACIONES FINALES
# ---------------------------------------

# 5.1 VALIDACIÓN DE FECHAS
# order_date <= shipped_date <= delivered_date

# Para pedidos con shipped_date, comprobamos que no sea anterior al order_date
assert (
    orders_df.dropna(subset=["shipped_date"])
    .eval("order_date <= shipped_date")
).all()

# Para pedidos con delivered_date, comprobamos que no sea anterior al shipped_date
assert (
    orders_df.dropna(subset=["delivered_date"])
    .eval("shipped_date <= delivered_date")
).all()

# 5.2 VALIDACIÓN DE ESTADOS DE PEDIDOS

# Un pedido pending NO debe tener delivered_date
assert orders_df.loc[
    orders_df["status"] == "pending",
    "delivered_date"
].isna().all()

# Un pedido cancelled NO debe tener delivered_date
assert orders_df.loc[
    orders_df["status"] == "cancelled",
    "delivered_date"
].isna().all()

# 5.3 VALIDACIÓN DE IMPORTES DE PAGOS

# Calculamos el total de cada pedido desde order_items
order_totals = (
    order_items_df
    .assign(total=lambda x: x["quantity"] * x["unit_price"] - x["discount_amount"])
    .groupby("order_id")["total"]
    .sum()
    .reset_index()
)

# Unimos con payments para comparar
payment_check = payments_df.merge(order_totals, on="order_id")

# Calculamos diferencia entre pago y total del pedido
payment_check["diff"] = abs(payment_check["amount"] - payment_check["total"])

# La diferencia debe ser prácticamente 0
assert (payment_check["diff"] < 0.01).all()

# 5.4 VALIDACIÓN DE CLAVES FORÁNEAS (foreign keys)

# order_items.order_id debe existir en orders
assert order_items_df["order_id"].isin(orders_df["order_id"]).all()

# order_items.product_id debe existir en products
assert order_items_df["product_id"].isin(products_df["product_id"]).all()

# payments.order_id debe existir en orders
assert payments_df["order_id"].isin(orders_df["order_id"]).all()

# reviews.order_item_id debe existir en order_items
assert reviews_df["order_item_id"].isin(order_items_df["order_item_id"]).all()


# 5.5 VALIDACIÓN DE REVIEWS SOLO EN PEDIDOS ENTREGADOS

review_check = (
    reviews_df
    .merge(order_items_df, on="order_item_id")
    .merge(orders_df, on="order_id")
)

# Todas las reviews deben pertenecer a pedidos con status = delivered
assert (review_check["status"] == "delivered").all()